In [4]:
!pip install deep_sort_realtime mediapipe opencv-python

  Using cached mediapipe-0.10.21-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached protobuf-4.25.8-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached sounddevice-0.5.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 20.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [1]:
!unzip /content/videos.zip

Archive:  /content/videos.zip
  inflating: 0LtLS9wROrk_E_000176_000204.mp4  
  inflating: 0LtLS9wROrk_E_000217_000300.mp4  
  inflating: 0LtLS9wROrk_E_000322_000417.mp4  
  inflating: 0LtLS9wROrk_E_000519_000594.mp4  
  inflating: 0LtLS9wROrk_E_000731_000738.mp4  
  inflating: 0LtLS9wROrk_E_000773_000802.mp4  
  inflating: 0LtLS9wROrk_E_000925_001024.mp4  
  inflating: 0MtilFKz4cA_E_001951_001980.mp4  
  inflating: 0MtilFKz4cA_E_002361_002435.mp4  


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from google.colab.patches import cv2_imshow
from deep_sort_realtime.deepsort_tracker import DeepSort

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    enable_segmentation=False,
    min_detection_confidence=0.5
)


mp_drawing = mp.solutions.drawing_utils


tracker = DeepSort(max_age=30, nn_budget=100, nms_max_overlap=1.0)


colors = np.random.randint(0, 255, size=(100, 3), dtype="uint8")


def process_frame(frame, pose, tracker):

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


    results = pose.process(rgb_frame)

    detections = []
    if results.pose_landmarks:

        landmarks = results.pose_landmarks.landmark
        xs = [lm.x * frame.shape[1] for lm in landmarks]
        ys = [lm.y * frame.shape[0] for lm in landmarks]

        x1, y1 = int(min(xs)), int(min(ys))
        x2, y2 = int(max(xs)), int(max(ys))
        w, h = x2 - x1, y2 - y1


        confidence = 0.9


        detections.append(([x1, y1, w, h], confidence, results.pose_landmarks))


    tracks = tracker.update_tracks(detections, frame=frame)


    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        bbox = track.to_ltrb()
        color = colors[int(track_id) % len(colors)].tolist()

        cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])),
                     (int(bbox[2]), int(bbox[3])), color, 2)
        cv2.putText(frame, f"ID: {track_id}", (int(bbox[0]), int(bbox[1]) - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        if len(detections) > 0 and detections[0][2]:
            mp_drawing.draw_landmarks(
                frame,
                detections[0][2],
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing.DrawingSpec(color=color, thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=color, thickness=2)
            )

    return frame


def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("Ошибка открытия видеофайла")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        processed_frame = process_frame(frame, pose, tracker)
        out.write(processed_frame)

        frame_count += 1
        if frame_count % 50 == 0:
            print(f"Обработано кадров: {frame_count}")

    cap.release()
    out.release()
    cv2.destroyAllWindows()


from os import listdir,path
from os.path import isfile, join
path_to_vids = '/content'
onlyfiles = [f for f in listdir(path_to_vids) if isfile(join(path_to_vids, f))]
for idx in range(len(onlyfiles)):
  process_video(path.join(path_to_vids,onlyfiles[idx]),path.join(path_to_vids,f'output_vid{idx}'))



Ошибка открытия видеофайла
Обработано кадров: 50
Обработано кадров: 100
